### import dependencies

In [1]:
from groq import Groq
import instructor
from qdrant_client import QdrantClient

from pydantic import BaseModel, Field



In [2]:
prompt="""
you are a helpful assistant.
return an answer to the question
Question: What is your name?
"""

In [14]:
client = Groq()
completion = client.chat.completions.create(
        model="qwen/qwen3-32b",  # qwen/qwen3-32b
        messages=[
        {
            "role": "system",
            "content": prompt
        }
        ],
        reasoning_effort="default",
        reasoning_format="hidden",
        temperature=0,
    )


In [15]:
print(completion.choices[0].message.content)

My name is Qwen. I am a large language model developed by Alibaba Cloud. How can I assist you today?


### add instructor {structured outputs}

In [20]:
client = instructor.from_provider("groq/qwen/qwen3-32b")

In [24]:
class RAGGenerationResponse(BaseModel):
    answer: str=Field(..., description="The answer to the question")

In [33]:
response,raw_response = client.create_with_completion(
    messages=[
        {
            "role": "system",
            "content": prompt
        }
        ],
        reasoning_effort="default",
        reasoning_format="hidden",
        temperature=0,
        response_model=RAGGenerationResponse,
)



In [34]:
response

RAGGenerationResponse(answer='Qwen')

In [35]:
raw_response

ChatCompletion(id='chatcmpl-6121457b-16ca-42a9-b1c1-16c8eec24327', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning=None, tool_calls=[ChatCompletionMessageToolCall(id='1jypcpb63', function=Function(arguments='{"answer":"Qwen"}', name='RAGGenerationResponse'), type='function')]))], created=1774881601, model='qwen/qwen3-32b', object='chat.completion', mcp_list_tools=None, service_tier='on_demand', system_fingerprint='fp_5cf921caa2', usage=CompletionUsage(completion_tokens=128, prompt_tokens=190, total_tokens=318, completion_time=0.213163676, completion_tokens_details=CompletionTokensDetails(reasoning_tokens=102), prompt_time=0.02282106, prompt_tokens_details=None, queue_time=0.1135956, total_time=0.235984736), usage_breakdown=None, x_groq=XGroq(id='req_01kmzjzx9wec1vfw82kngj1qp3', debug=None, seed=2126960360, usage=None))

In [36]:
class RAGGenerationResponse(BaseModel):
    answer: str=Field(..., description="The answer to the question")
    reasoning_trace: str=Field(..., description="The reasoning trace for the answer")

In [37]:
response,raw_response = client.create_with_completion(
    messages=[
        {
            "role": "system",
            "content": prompt
        }
        ],
        reasoning_effort="default",
        reasoning_format="hidden",
        temperature=0,
        response_model=RAGGenerationResponse,
)



In [38]:
response

RAGGenerationResponse(answer='Qwen', reasoning_trace="The user asked for my name, so I used the RAGGenerationResponse function to return 'Qwen' as the answer. The reasoning trace demonstrates the correct application of the provided tool for this query.")

In [40]:
client = instructor.from_provider("groq/qwen/qwen3-32b")

In [44]:
QDRANT_URL = "http://localhost:6333"

In [59]:
import os

from langchain_huggingface import HuggingFaceEndpointEmbeddings


# Environment variables
HF_API_TOKEN = os.environ.get("HF_API_TOKEN")
QDRANT_URL = os.environ.get("QDRANT_URL", "http://localhost:6333")

# Model Configuration
HF_EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
GROQ_LLM_MODEL = "qwen/qwen3-32b"

In [60]:
model = "sentence-transformers/all-MiniLM-L6-v2"
hf = HuggingFaceEndpointEmbeddings(
            model=model,
            huggingfacehub_api_token=os.environ["HF_API_TOKEN"],
        )



c:\Users\Loq\Documents\CRAP\end to end aibootcamp\code\handsON\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [61]:
from groq import Groq
from qdrant_client import QdrantClient
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen
import json


def _mean_pool_embedding(raw_embedding):
    if not raw_embedding:
        raise ValueError("Hugging Face API returned an empty embedding.")

    if isinstance(raw_embedding[0], (int, float)):
        return [float(value) for value in raw_embedding]

    if isinstance(raw_embedding[0], list):
        token_count = len(raw_embedding)
        vector_size = len(raw_embedding[0])
        pooled = [0.0] * vector_size

        for token_vector in raw_embedding:
            if len(token_vector) != vector_size:
                raise ValueError("Inconsistent token vector dimensions in HF embedding response.")
            for idx, value in enumerate(token_vector):
                pooled[idx] += float(value)

        return [value / token_count for value in pooled]

    raise ValueError("Unexpected Hugging Face embedding response format.")

def get_embedding(text, model_name: str | None = None):
    """
    Embed text using HuggingFace Embeddings API.
    Model: sentence-transformers/all-MiniLM-L6-v2
    """
    selected_model = model_name or HF_EMBEDDING_MODEL
    endpoint = (
        "https://router.huggingface.co/hf-inference/models/"
        f"{selected_model}/pipeline/feature-extraction"
    )
    payload = json.dumps(
        {
            "inputs": text,
            "normalize": True,
        }
    ).encode("utf-8")

    headers = {"Content-Type": "application/json"}
    if HF_API_TOKEN:
        headers["Authorization"] = f"Bearer {HF_API_TOKEN}"

    request = Request(endpoint, data=payload, headers=headers, method="POST")

    try:
        with urlopen(request, timeout=60) as response:
            response_data = json.loads(response.read().decode("utf-8"))
    except HTTPError as exc:
        message = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(
            f"Hugging Face embedding API request failed ({exc.code}): {message}"
        ) from exc
    except URLError as exc:
        raise RuntimeError(f"Could not reach Hugging Face embedding API: {exc}") from exc

    if isinstance(response_data, dict) and response_data.get("error"):
        raise RuntimeError(f"Hugging Face embedding API error: {response_data['error']}")

    if isinstance(response_data, list) and len(response_data) == 1:
        return _mean_pool_embedding(response_data[0])
    

    return _mean_pool_embedding(response_data)

def retrieve_data(query, qdrant_client, top_k=5):
    """
    Retrieve similar documents from Qdrant vector database.
    
    Args:
        query (str): User question
        qdrant_client: Qdrant client instance
        top_k (int): Number of documents to retrieve
    
    Returns:
        dict: Retrieved context with IDs, descriptions, ratings, and similarity scores
    """
    query_embedding = get_embedding(query)

    search_result = qdrant_client.query_points(
        collection_name="amazon-items-collection-00",
        query=query_embedding,
        limit=top_k,
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []
    for search_result in search_result.points:
        retrieved_context_ids.append(search_result.payload["parent_asin"])
        retrieved_context.append(search_result.payload["description"])
        retrieved_context_ratings.append(search_result.payload["average_rating"])
        similarity_scores.append(search_result.score)

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "retrieved_context_ratings": retrieved_context_ratings,
        "similarity_scores": similarity_scores,
    }

def process_context(context):
    """Format retrieved context for LLM prompt."""
    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"
    return formatted_context


def build_prompt(preprocessed_context, question):
    """Build the prompt for the LLM."""
    prompt = f"""
    You are a shopping assistant that can answer questions about the products in the stock.

    You will be given a question and a list of context.

    Instructions:
    -You need to answer the question based on the provided context only.
    -never use the word context and refer to it as the available products.

    Context:    
    {preprocessed_context}

    Question: {question}
""" 
    return prompt

# Initialize Groq client
#client = Groq()

def generate_answer(prompt):
    """
    Generate answer using Groq LLM.
    
    Model: qwen/qwen3-32b
    
    Args:
        prompt (str): The formatted prompt with context and question
        
    Returns:
        str: Generated answer from the LLM
    """
    completion,raw_response = client.create_with_completion(
        messages=[
        {
            "role": "system",
            "content": prompt
        }
        ],
        reasoning_effort="default",
        reasoning_format="hidden",
        temperature=0,
        response_model=RAGGenerationResponse,
    )
    return completion

def rag_pipeline(query,qdrant_client, top_k=5):
    """
    Complete RAG Pipeline.
    
    Models Used:
    - Embedding: HuggingFace (sentence-transformers/all-MiniLM-L6-v2)
    - LLM: Groq (qwen/qwen3-32b)
    - Vector DB: Qdrant
    
    Args:
        query (str): User question
        top_k (int): Number of context documents to retrieve
        
    Returns:
        dict: {
            "answer": str - Generated answer,
            "question": str - Original question,
            "retrieved_context_ids": list - IDs of retrieved documents,
            "retrieved_context": list - Content of retrieved documents,
            "similarity_scores": list - Relevance scores
        }
    """
    #qdrant_client = QdrantClient(url=QDRANT_URL)

    retrieved_context = retrieve_data(query, qdrant_client, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, query)
    answer = generate_answer(prompt)

    final_result = {
        "datamodel":answer,
        "answer": answer.answer,
        "question": query,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"],
        "similarity_scores": retrieved_context["similarity_scores"],
    }

    return final_result

In [62]:
qdrant_client=QdrantClient(url="http://localhost:6333")

In [63]:
output = rag_pipeline("What are the best romantic albums i can get?",qdrant_client)

In [64]:
output["answer"]

"The best romantic albums available are:\n\n1. **I'm Wide Awake, It's Morning by Bright Eyes** (ID: B0B97ZQZBK) - Features 'First Day of My Life,' voted the #1 love song of all time by NPR Music, and captures heartfelt, vulnerable themes.\n\n2. **MERCY by John Cale** (ID: B0BJP8ZBRT) - Includes vulnerable love songs like 'EVERLASTING DAYS' and collaborations with artists like Weyes Blood, blending emotional depth with reinvented musical styles.\n\n3. **Blue Rev by Alvvays** (ID: B0B65SZKLJ) - Explores romantic tension and lovelorn confusion in tracks like 'Velveteen' and 'Tile by Tile,' balancing playfulness with emotional resonance.\n\nFor a more instrumental approach, **NIGHT by George Winston** (ID: B09SJ3BCLJ) offers a serene, nocturnal atmosphere with a cover of Leonard Cohen’s 'Hallelujah,' evoking romantic introspection."